# 東京23区の坪単価推移分析（2005-2025年）

Prophet時系列分析の詳細可視化：

- 各区ごとの平均・中央値推移
- 各取引データの散布図
- 時系列トレンドの比較


In [2]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

## 1. データ読み込み


In [3]:
# データ読み込み
df = pl.read_csv("../data/ml_dataset/tokyo_23_ml_dataset.csv")

# transaction_dateをDatetime型に変換
df = df.with_columns(pl.col("transaction_date").str.to_datetime())

print(f"総データ件数: {len(df):,}件")
print(f"期間: {df['transaction_date'].min()} 〜 {df['transaction_date'].max()}")
print(f"\nカラム: {df.columns}")

総データ件数: 325,042件
期間: 2005-01-01 00:00:00 〜 2025-01-01 00:00:00

カラム: ['transaction_date', 'Year', 'tsubo_price', 'Area', 'TotalFloorArea', 'Age', 'BuildingYear', 'FloorPlan', 'TimeToNearestStation', 'NearestStation', 'DistrictName', 'CoverageRatio', 'FloorAreaRatio', 'Renovation', 'Municipality', 'Structure', 'CityPlanning']


## 2. 東京23区のデータをフィルタリング


In [4]:
# 東京23区のリスト
tokyo_23_wards = [
    "千代田区",
    "中央区",
    "港区",
    "新宿区",
    "文京区",
    "台東区",
    "墨田区",
    "江東区",
    "品川区",
    "目黒区",
    "大田区",
    "世田谷区",
    "渋谷区",
    "中野区",
    "杉並区",
    "豊島区",
    "北区",
    "荒川区",
    "板橋区",
    "練馬区",
    "足立区",
    "葛飾区",
    "江戸川区",
]

# 23区データのフィルタリング
df_tokyo = df.filter(pl.col("Municipality").is_in(tokyo_23_wards))

print(f"東京23区のデータ件数: {len(df_tokyo):,}件")
print(f"\n各区のデータ件数:")
print(df_tokyo.group_by("Municipality").agg(pl.len()).sort("len", descending=True))

東京23区のデータ件数: 325,042件

各区のデータ件数:
shape: (23, 2)
┌──────────────┬───────┐
│ Municipality ┆ len   │
│ ---          ┆ ---   │
│ str          ┆ u32   │
╞══════════════╪═══════╡
│ 江東区       ┆ 25614 │
│ 大田区       ┆ 23258 │
│ 世田谷区     ┆ 22242 │
│ 港区         ┆ 20978 │
│ 新宿区       ┆ 19464 │
│ …            ┆ …     │
│ 葛飾区       ┆ 8895  │
│ 中野区       ┆ 8873  │
│ 北区         ┆ 8868  │
│ 荒川区       ┆ 6465  │
│ 千代田区     ┆ 5900  │
└──────────────┴───────┘


## 3. 時系列データの集約（月単位）


In [5]:
# 月単位で集約
df_monthly = (
    df_tokyo.sort("transaction_date")
    .group_by_dynamic("transaction_date", every="1mo", group_by="Municipality")
    .agg(
        [
            pl.col("tsubo_price").mean().alias("mean_price"),
            pl.col("tsubo_price").median().alias("median_price"),
            pl.col("tsubo_price").count().alias("transaction_count"),
        ]
    )
    .sort(["Municipality", "transaction_date"])
)

print(df_monthly.head(10))

shape: (10, 5)
┌──────────────┬─────────────────────┬────────────┬──────────────┬───────────────────┐
│ Municipality ┆ transaction_date    ┆ mean_price ┆ median_price ┆ transaction_count │
│ ---          ┆ ---                 ┆ ---        ┆ ---          ┆ ---               │
│ str          ┆ datetime[μs]        ┆ f64        ┆ f64          ┆ u32               │
╞══════════════╪═════════════════════╪════════════╪══════════════╪═══════════════════╡
│ 世田谷区     ┆ 2005-01-01 00:00:00 ┆ 2.2760e6   ┆ 2.1543e6     ┆ 234               │
│ 世田谷区     ┆ 2006-01-01 00:00:00 ┆ 2.1567e6   ┆ 2.0411e6     ┆ 528               │
│ 世田谷区     ┆ 2007-01-01 00:00:00 ┆ 2.2815e6   ┆ 2.2039e6     ┆ 485               │
│ 世田谷区     ┆ 2008-01-01 00:00:00 ┆ 2.3913e6   ┆ 2.3140e6     ┆ 542               │
│ 世田谷区     ┆ 2009-01-01 00:00:00 ┆ 2.1484e6   ┆ 2.1332e6     ┆ 644               │
│ 世田谷区     ┆ 2010-01-01 00:00:00 ┆ 2.4773e6   ┆ 2.3508e6     ┆ 802               │
│ 世田谷区     ┆ 2011-01-01 00:00:00 ┆ 2.2682e6   ┆ 2.16

## 4. 可視化: 23区ごとの坪単価推移（平均・中央値・散布図）


In [6]:
# Plotlyで可視化（全区を1つのグラフに）
fig = go.Figure()

# 各区ごとにトレースを追加
for ward in tokyo_23_wards:
    # 月次集約データ
    df_ward_monthly = df_monthly.filter(pl.col("Municipality") == ward)

    if len(df_ward_monthly) == 0:
        continue

    # 平均値の線
    fig.add_trace(
        go.Scatter(
            x=df_ward_monthly["transaction_date"].to_list(),
            y=df_ward_monthly["mean_price"].to_list(),
            mode="lines",
            name=f"{ward} (平均)",
            line=dict(width=2),
            legendgroup=ward,
        )
    )

    # 中央値の線
    fig.add_trace(
        go.Scatter(
            x=df_ward_monthly["transaction_date"].to_list(),
            y=df_ward_monthly["median_price"].to_list(),
            mode="lines",
            name=f"{ward} (中央値)",
            line=dict(width=1, dash="dot"),
            legendgroup=ward,
            visible="legendonly",  # デフォルトで非表示
        )
    )

# 各点の散布図（サンプリングして表示）
df_sample = df_tokyo.sample(n=min(10000, len(df_tokyo)), seed=42)  # 最大1万点まで

for ward in tokyo_23_wards:
    df_ward_points = df_sample.filter(pl.col("Municipality") == ward)

    if len(df_ward_points) == 0:
        continue

    fig.add_trace(
        go.Scatter(
            x=df_ward_points["transaction_date"].to_list(),
            y=df_ward_points["tsubo_price"].to_list(),
            mode="markers",
            name=f"{ward} (各点)",
            marker=dict(size=3, opacity=0.3),
            legendgroup=ward,
            visible="legendonly",  # デフォルトで非表示
        )
    )

# レイアウト設定
fig.update_layout(
    title="東京23区の坪単価推移（2005-2025年）",
    xaxis_title="年月",
    yaxis_title="坪単価（円/坪）",
    height=800,
    hovermode="x unified",
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
)

fig.show()

## 5. 区ごとの個別グラフ（詳細表示）


In [7]:
# 特定の区を選択して詳細表示
selected_wards = df_tokyo.group_by("Municipality").agg(pl.len())["Municipality"].to_list()

for ward in selected_wards:
    # 月次集約データ
    df_ward_monthly = df_monthly.filter(pl.col("Municipality") == ward)

    # 全データ点
    df_ward_all = df_tokyo.filter(pl.col("Municipality") == ward)

    if len(df_ward_monthly) == 0:
        continue

    # グラフ作成
    fig = go.Figure()

    # 散布図（全データ点）
    # fig.add_trace(go.Scatter(
    #     x=df_ward_all["transaction_date"].to_list(),
    #     y=df_ward_all["tsubo_price"].to_list(),
    #     mode="markers",
    #     name="各取引データ",
    #     marker=dict(size=4, opacity=0.2, color="lightblue"),
    # ))

    # 平均値
    fig.add_trace(
        go.Scatter(
            x=df_ward_monthly["transaction_date"].to_list(),
            y=df_ward_monthly["mean_price"].to_list(),
            mode="lines+markers",
            name="月次平均",
            line=dict(width=3, color="red"),
            marker=dict(size=6),
        )
    )

    # 中央値
    fig.add_trace(
        go.Scatter(
            x=df_ward_monthly["transaction_date"].to_list(),
            y=df_ward_monthly["median_price"].to_list(),
            mode="lines+markers",
            name="月次中央値",
            line=dict(width=3, color="blue", dash="dash"),
            marker=dict(size=6),
        )
    )

    fig.update_layout(
        title=f"{ward}の坪単価推移（2005-2025年）",
        xaxis_title="年月",
        yaxis_title="坪単価（円/坪）",
        height=600,
        hovermode="x unified",
    )

    fig.show()

## 6. 統計サマリー


In [8]:
# 区ごとの統計サマリー
summary = (
    df_tokyo.group_by("Municipality")
    .agg(
        [
            pl.col("tsubo_price").mean().alias("mean_price"),
            pl.col("tsubo_price").median().alias("median_price"),
            pl.col("tsubo_price").std().alias("std_price"),
            pl.col("tsubo_price").min().alias("min_price"),
            pl.col("tsubo_price").max().alias("max_price"),
            pl.col("tsubo_price").count().alias("transaction_count"),
        ]
    )
    .sort("mean_price", descending=True)
)

print("\n東京23区の坪単価統計サマリー（2005-2025年）")
print(summary)


東京23区の坪単価統計サマリー（2005-2025年）
shape: (23, 7)
┌──────────────┬────────────┬──────────────┬──────────────┬──────────────┬───────────┬─────────────┐
│ Municipality ┆ mean_price ┆ median_price ┆ std_price    ┆ min_price    ┆ max_price ┆ transaction │
│ ---          ┆ ---        ┆ ---          ┆ ---          ┆ ---          ┆ ---       ┆ _count      │
│ str          ┆ f64        ┆ f64          ┆ f64          ┆ f64          ┆ f64       ┆ ---         │
│              ┆            ┆              ┆              ┆              ┆           ┆ u32         │
╞══════════════╪════════════╪══════════════╪══════════════╪══════════════╪═══════════╪═════════════╡
│ 港区         ┆ 4.5241e6   ┆ 3.9669e6     ┆ 2.8273e6     ┆ 115702.47933 ┆ 1.3223e8  ┆ 20978       │
│              ┆            ┆              ┆              ┆ 9            ┆           ┆             │
│ 千代田区     ┆ 4.1487e6   ┆ 3.7190e6     ┆ 2.3414e6     ┆ 158677.68595 ┆ 7.1865e7  ┆ 5900        │
│ 渋谷区       ┆ 3.9406e6   ┆ 3.4947e6     ┆ 2.1418e6   

## 7. 年次トレンドの比較


In [9]:
# 年次集約
df_yearly = (
    df_tokyo.with_columns(pl.col("transaction_date").dt.year().alias("year"))
    .group_by(["Municipality", "year"])
    .agg(
        [
            pl.col("tsubo_price").mean().alias("mean_price"),
            pl.col("tsubo_price").median().alias("median_price"),
            pl.col("tsubo_price").count().alias("transaction_count"),
        ]
    )
    .sort(["Municipality", "year"])
)

# ヒートマップで可視化
pivot_data = df_yearly.pivot(values="mean_price", index="Municipality", on="year")

# Plotly Heatmap
fig = go.Figure(
    data=go.Heatmap(
        z=pivot_data.select(pl.exclude("Municipality")).to_numpy(),
        x=[str(col) for col in pivot_data.columns if col != "Municipality"],
        y=pivot_data["Municipality"].to_list(),
        colorscale="Viridis",
        colorbar=dict(title="平均坪単価（円/坪）"),
    )
)

fig.update_layout(
    title="東京23区の年次平均坪単価ヒートマップ（2005-2025年）", xaxis_title="年", yaxis_title="区", height=800
)

fig.show()

## 8. Prophet予測精度の確認

区ごとにProphetモデルを学習し、2005-2023年のデータでトレーニング、2024-2025年のデータで予測精度を確認します。


In [ ]:
# 区ごとの評価指標の比較

wards_list = list(prophet_results.keys())
mae_list = [prophet_results[w]["mae"] for w in wards_list]
rmse_list = [prophet_results[w]["rmse"] for w in wards_list]
mape_list = [prophet_results[w]["mape"] for w in wards_list]

# サブプロットで3つの指標を表示
fig = make_subplots(rows=1, cols=3, subplot_titles=("MAE (円/坪)", "RMSE (円/坪)", "MAPE (%)"), horizontal_spacing=0.12)

# MAE
fig.add_trace(go.Bar(x=wards_list, y=mae_list, name="MAE", marker_color="lightblue"), row=1, col=1)

# RMSE
fig.add_trace(go.Bar(x=wards_list, y=rmse_list, name="RMSE", marker_color="lightcoral"), row=1, col=2)

# MAPE
fig.add_trace(go.Bar(x=wards_list, y=mape_list, name="MAPE", marker_color="lightgreen"), row=1, col=3)

fig.update_layout(title_text="Prophet予測精度の区ごと比較", height=500, showlegend=False)

fig.update_xaxes(tickangle=45)

fig.show()

In [ ]:
# 区ごとの時系列予測グラフ

for ward, result in prophet_results.items():
    fig = go.Figure()

    # 実測値
    fig.add_trace(
        go.Scatter(
            x=result["test_dates"],
            y=result["y_true"],
            mode="markers",
            name="実測値",
            marker=dict(size=5, color="blue", opacity=0.5),
        )
    )

    # 予測値
    fig.add_trace(
        go.Scatter(
            x=result["test_dates"],
            y=result["y_pred"],
            mode="markers",
            name="Prophet予測値",
            marker=dict(size=5, color="red", opacity=0.5),
        )
    )

    fig.update_layout(
        title=f"{ward} - 2024-2025年の予測精度 (MAE: {result['mae']:,.0f}円/坪, MAPE: {result['mape']:.2f}%)",
        xaxis_title="取引日",
        yaxis_title="坪単価（円/坪）",
        height=500,
        hovermode="x unified",
    )

    fig.show()

In [ ]:
# Prophet予測結果の可視化：実測値 vs 予測値（散布図）

fig = go.Figure()

for ward, result in prophet_results.items():
    fig.add_trace(
        go.Scatter(
            x=result["y_true"],
            y=result["y_pred"],
            mode="markers",
            name=ward,
            marker=dict(size=5, opacity=0.6),
            text=[f"{ward}<br>実測: {y:.0f}<br>予測: {p:.0f}" for y, p in zip(result["y_true"], result["y_pred"])],
            hovertemplate="%{text}<extra></extra>",
        )
    )

# 理想的な予測線（y=x）
max_val = max([result["y_true"].max() for result in prophet_results.values()])
min_val = min([result["y_true"].min() for result in prophet_results.values()])

fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        name="理想的な予測（y=x）",
        line=dict(color="black", dash="dash", width=2),
    )
)

fig.update_layout(
    title="Prophet予測精度: 実測値 vs 予測値（2024-2025年テストデータ）",
    xaxis_title="実測坪単価（円/坪）",
    yaxis_title="予測坪単価（円/坪）",
    height=700,
    width=900,
)

fig.show()

In [ ]:
# 主要な区を選択してProphet予測を実行
selected_wards_prophet = ["港区", "千代田区", "渋谷区", "世田谷区", "足立区", "葛飾区"]

prophet_results = {}

for ward in selected_wards_prophet:
    print(f"\n{'=' * 60}")
    print(f"{ward} - Prophet予測")
    print("=" * 60)

    # 学習データとテストデータを区で絞り込み
    train_ward = train_df.filter(pl.col("Municipality") == ward)
    test_ward = test_df.filter(pl.col("Municipality") == ward)

    # 月単位で集約（学習データ）
    train_monthly = (
        train_ward.group_by_dynamic("transaction_date", every="1mo")
        .agg(pl.col("tsubo_price").mean().alias("y"))
        .sort("transaction_date")
        .rename({"transaction_date": "ds"})
    )

    # Polars -> Pandas変換
    train_prophet = train_monthly.to_pandas()

    print(f"学習データ: {len(train_prophet)}月分")

    # Prophetモデル作成
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
    )
    model.add_country_holidays(country_name="JP")

    # 学習
    model.fit(train_prophet)

    # テストデータの予測
    # データポイントレベルで予測するため、test_wardの日付を使用
    test_dates = pd.DataFrame({"ds": test_ward["transaction_date"].to_pandas()})
    forecast = model.predict(test_dates)

    # 実測値と予測値を比較
    y_true = test_ward["tsubo_price"].to_numpy()
    y_pred = forecast["yhat"].values

    # 評価指標
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"MAE: {mae:,.0f}円/坪")
    print(f"RMSE: {rmse:,.0f}円/坪")
    print(f"MAPE: {mape:.2f}%")

    # 結果を保存
    prophet_results[ward] = {
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "y_true": y_true,
        "y_pred": y_pred,
        "test_dates": test_ward["transaction_date"].to_list(),
        "forecast": forecast,
    }

print("\n" + "=" * 60)
print("全体サマリー")
print("=" * 60)
for ward, metrics in prophet_results.items():
    print(
        f"{ward:12s} - MAE: {metrics['mae']:>10,.0f}円/坪, RMSE: {metrics['rmse']:>10,.0f}円/坪, MAPE: {metrics['mape']:>6.2f}%"
    )

In [ ]:
# Prophetのインポート
from prophet import Prophet
import pandas as pd
import numpy as np

# データを2005-2023年（学習）と2024-2025年（テスト）に分割
train_df = df_tokyo.filter(pl.col("transaction_date") < pl.datetime(2024, 1, 1))
test_df = df_tokyo.filter(pl.col("transaction_date") >= pl.datetime(2024, 1, 1))

print(f"学習データ: {len(train_df):,}件 ({train_df['transaction_date'].min()} - {train_df['transaction_date'].max()})")
print(f"テストデータ: {len(test_df):,}件 ({test_df['transaction_date'].min()} - {test_df['transaction_date'].max()})")